In [ ]:
!wget -q https://www.cs.ucsb.edu/~william/data/liar_dataset.zip -O liar_dataset.zip
!unzip -q -o liar_dataset.zip

import pandas as pd, os, collections

colnames = [
    "id", "label", "statement", "subject", "speaker", "job_title",
    "state_info", "party_affiliation",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts",
    "context"
]

train_df = pd.read_csv("train.tsv", sep="\t", header=None, names=colnames, quoting=3)
valid_df = pd.read_csv("valid.tsv", sep="\t", header=None, names=colnames, quoting=3)
test_df  = pd.read_csv("test.tsv",  sep="\t", header=None, names=colnames, quoting=3)

false_set = {"pants-fire", "false", "barely-true"}
true_set  = {"half-true", "mostly-true", "true"}

def to_binary(df):
    df["label_norm"] = df["label"].astype(str).str.strip().str.lower()
    df["labels"] = df["label_norm"].apply(lambda s: 1 if s in true_set else 0)  # 1=true, 0=false
    ### 只保留 statement +labels
    return df[["statement","labels"]]

train_df_b = to_binary(train_df)
valid_df_b = to_binary(valid_df)
test_df_b  = to_binary(test_df)

def stats(df, name):
    cnt = dict(collections.Counter(df["labels"].astype(int).tolist()))
    print(f"{name:5s} | n={len(df):4d} | class balance (0/1) = {cnt}")

stats(train_df_b, "train")
stats(valid_df_b, "valid")
stats(test_df_b,  "test")

# 看 3 条样例（列名：statement / labels）
train_df_b.head(3)

train | n=10269 | class balance (0/1) = {0: 4497, 1: 5772}
valid | n=1284 | class balance (0/1) = {0: 616, 1: 668}
test  | n=1283 | class balance (0/1) = {1: 727, 0: 556}


,statement,labels
0,Says the Annies List political group supports ...,0
1,When did the decline of coal start? It started...,1
2,"Hillary Clinton agrees with John McCain ""by vo...",1


In [ ]:
import time
from sklearn.metrics import f1_score, accuracy_score

def metric_MacroF1_calc(true_labels_list, predicted_labels_list):
    """
    calc Macro-F1, main metric：
    """
    return f1_score(true_labels_list, predicted_labels_list, average="macro")

def metric_Accuracy_calc(true_labels_list, predicted_labels_list):
    """
    calc Accuracy, secondary metrics
    """
    return accuracy_score(true_labels_list, predicted_labels_list)

def metric_latency_calc(text2text_generator_pipeline, input_texts, n_samples_for_timing=300, max_new_tokens=4):
    """
    return mean each_sample_time(ms), float
    """
    total_to_measure = min(n_samples_for_timing, len(input_texts))

    # 热身：第一次生成常常包含编译/缓存，慢且不稳定，不计入统计
    warmup_output = text2text_generator_pipeline(input_texts[0], max_new_tokens=max_new_tokens)

    start_time = time.time()
    for i in range(total_to_measure):
        generated_output = text2text_generator_pipeline(input_texts[i], max_new_tokens=max_new_tokens)
        # 这里只是计时，不需要用 generated_output 的内容
    avg_ms_per_sample = (time.time() - start_time) / total_to_measure * 1000.0
    return avg_ms_per_sample

'''！！！！！！！！！！！！！！！！！'''
'''要注意万一生成式ai生成模型可能偶尔吐出别的词（比如 “barely-true”，怎么办？？
暂且来看，只会是true/false对吧？到时候真生成了乱七八糟的我再想办法,再加限制就行。
Limit max_new_token, enhance paraphrase ability'''

def map_generated_label_text_to_binary_label(generated_label):
    text_lower = str(generated_label).strip().lower()
    return 1 if ("true" in text_lower) else 0

In [ ]:
    ''' The universal config for every model, these are the main config
        the others we use defualt
    '''
Config_Universal = dict(
    max_input_length=256,
    max_new_tokens=64,   # if == 4, only generate "True"/"False"
    random_seed=42,
    train_epochs=3,
    batch_size_train=8,
    batch_size_eval=16,
    gradient_accumulation_steps=4,
    sampling_on = False
)

def get_generation_kwargs_from_config(config_dict):
  sampling_on = config_dict["sampling_on"]
  if sampling_on:
        return dict( do_sample = True,
          temperature = 1.0, top_k = 50, top_p = 0.9,
          max_new_tokens = config_dict["max_new_tokens"],
          truncation = True,
          max_length = config_dict["max_input_length"],
          num_beams=1 )


  return dict( do_sample =False,
    max_new_tokens = config_dict["max_new_tokens"],
    truncation=True,
    max_length = config_dict["max_input_length"],
    num_beams=1 )

In [ ]:

def prompt_setting(statement_text, setting_name):
    """
    - baseline: only statement given
    - instruction only: instruction + statement（Let model know its task）, make output True/False
    - CoT: instruction + CoT + statement. Short Rational first, then label output.
    """
    statement_text = str(statement_text)
    setting_name= str(setting_name).strip()

    # baseline: 只喂原句
    if setting_name == "baseline":
        return statement_text

    # instruction only: instruction + statement（Let model know its task）
    if setting_name == "instruction":
      return ("Instruction: Decide whether the following claim is true or false.\n"
          f'Statement: "{statement_text}"\n'
          "Options: True or False.\n"
          "Output the label only.")

    # instruction + CoT: instruction + CoT + statement（Let model know its task）
    if setting_name == "CoT":
        return ("Instruction: Decide whether the following claim is true or false.\n"
            "Before the final decision, briefly explain your reasoning in 1-2 sentences.\n"
            f'Statement: \"{statement_text}\"\n'
            "Options: True or False.\n"
            "Format:\n"
            "Reasoning: <short explanation>\n"
            "Final label: <True or False>.")

    raise ValueError("setting_name must be 'baseline', 'instruction', or 'CoT'.")


def build_TF_from_label(binary_label_int):
    # In label we used 1=true, 0=false but for model we use "True"/"False"
    return "True" if int(binary_label_int) == 1 else "False"


In [ ]:


##### Prompt Test #############
test_case = test_df_b["statement"].iloc[2]
test_settings = ["baseline", "instruction", "CoT", "WHAT???"]
print( f"Statement: {test_case}\n")
print()
for setting_name in test_settings:
    print(f"--- prompt '{setting_name}' ---")
    try:
        prompt_text = prompt_setting(test_case, setting_name)
        print(prompt_text)
    except ValueError as err:
        # demonstration of illegal input (passing wrong prompt setting )
        print(f"[Error]: {err}")
    print()
print(test_df_b)


Statement: Says John McCain has done nothing to help the vets.


--- prompt 'baseline' ---
Says John McCain has done nothing to help the vets.

--- prompt 'instruction' ---
Instruction: Decide whether the following claim is true or false.
Statement: "Says John McCain has done nothing to help the vets."
Options: True or False.
Output the label only.

--- prompt 'CoT' ---
Instruction: Decide whether the following claim is true or false.
Before the final decision, briefly explain your reasoning in 1-2 sentences.
Statement: "Says John McCain has done nothing to help the vets."
Options: True or False.
Format:
Reasoning: <short explanation>
Final label: <True or False>.

--- prompt 'WHAT???' ---
[Error]: setting_name must be 'baseline', 'instruction', or 'CoT'.

                                              statement  labels
0     Building a wall on the U.S.-Mexico border will...       1
1     Wisconsin is on pace to double the number of l...       0
2     Says John McCain has done nothing t

In [ ]:
# select model
MODEL_NAME = "google/flan-t5-small"
#MODEL_NAME = "facebook/bart-base"

import numpy as np, torch, random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

'''
# ==== fix random seed====
def set_global_seed(seed_value: int):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_global_seed(Config_Universal["random_seed"])
'''

#======== 加载 tokenizer / model / pipeline ====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)



# ==== Gnerate Key arguments for model from Config_Universal====
gen_kwargs = get_generation_kwargs_from_config(Config_Universal)

print("Device:", device)
print("Loaded:", MODEL_NAME)
print("Generation kwargs:", gen_kwargs)


Device: cuda
Loaded: google/flan-t5-small
Generation kwargs: {'do_sample': False, 'max_new_tokens': 64, 'truncation': True, 'max_length': 256, 'num_beams': 1}


In [ ]:
#=============进行烟雾测试（try some prompts)============


# 准备少量 prompts，前三个
instruction_prompts = []
cot_prompts = []

for s in test_df_b["statement"].head(3):
    instruction_prompts.append(prompt_setting(s, "instruction"))
    cot_prompts.append(prompt_setting(s, "CoT"))

# control the process of converting a string into a tensor (how much to truncate, whether to padding, and what tensor format to return).
tokenizer_kwargs = dict(
    padding = False,
    truncation = True,
    max_length = Config_Universal["max_input_length"],
    return_tensors = "pt",
)

# control how the model output is generated (greedy/sampling/beam, how many new tokens to generate)
generate_kwargs = dict(
    do_sample=False,    #greedy；sampling时改为 True 并加 top_p/top_k/temperature
    num_beams=1,
    max_new_tokens=Config_Universal["max_new_tokens"],
)

def generate_outputs_for_prompts(prompts_list):
    outputs_text = []
    for prompt_text in prompts_list:
        print(f"{prompt_text}\n")
        encoded = tokenizer(prompt_text, **tokenizer_kwargs).to(device)

        print(f"ENCODED:\n{encoded}\n")


        with torch.no_grad():
            generated_ids = model.generate(**encoded, **generate_kwargs)
        decoded_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        outputs_text.append(decoded_text)
    return outputs_text

instruction_outputs = generate_outputs_for_prompts(instruction_prompts)
cot_outputs = generate_outputs_for_prompts(cot_prompts)

# 映射并打印
instruction_preds = []
for t in instruction_outputs:
    instruction_preds.append(map_generated_label_text_to_binary_label(t))

cot_preds = []
for t in cot_outputs:
    cot_preds.append(map_generated_label_text_to_binary_label(t))

for i in range(3):
    print(f"\n=== Instruction Example #{i} ===")
    print("PROMPT:\n", instruction_prompts[i])
    print("OUTPUT:\n", instruction_outputs[i])
    print("MAPPED LABEL:", instruction_preds[i])

for i in range(3):
    print(f"\n=== CoT Example #{i} ===")
    print("PROMPT:\n", cot_prompts[i])
    print("OUTPUT:\n", cot_outputs[i])
    print("MAPPED LABEL:", cot_preds[i])


Instruction: Decide whether the following claim is true or false.
Statement: "Building a wall on the U.S.-Mexico border will take literally years."
Options: True or False.
Output the label only.

ENCODED:
{'input_ids': tensor([[21035,    10, 14600,   221,   823,     8,   826,  1988,    19,  1176,
            42,  6136,     5, 16836,    10,    96, 24752,    53,     3,     9,
          1481,    30,     8,   412,     5,   134,     5,    18,   329,   994,
          5807,  4947,    56,   240,  6672,   203,   535, 17011,    10, 10998,
            42, 10747,     7,    15,     5,  3387,  2562,     8,  3783,   163,
             5,     1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]], device='cuda:0')}

Instruction: Decide whether the following claim is true or false.
Statement: "Wisconsin is on pace to double the number o